<h1>TASK-1</h1>

In [1]:
import pandas as pd
import numpy as np

np.random.seed(42)

n = 500000

cities = ["Berlin","Tokyo","New York","Paris","London","Dubai","Toronto","Sydney","Madrid","Rome"]

df = pd.DataFrame({
    "user_id": np.arange(1, n + 1),
    "city": np.random.choice(cities, n),
    "score": np.random.uniform(0, 100, n),
    "active": np.random.choice([True, False], n),
    "signup_date": pd.to_datetime("today") - pd.to_timedelta(np.random.randint(0, 365*3, n), unit="D"),
    "age": np.random.randint(18, 81, n),
    "sessions": np.random.randint(0, 501, n),
    "revenue": np.random.uniform(0, 1000, n)
})

print(df.head())
print(len(df))



   user_id     city      score  active                signup_date  age  \
0        1  Toronto  43.689490   False 2026-02-05 21:47:54.562327   71   
1        2    Paris  97.403462    True 2025-06-06 21:47:54.562327   39   
2        3   Sydney  66.148238   False 2025-11-12 21:47:54.562327   23   
3        4   London  21.993544   False 2025-05-26 21:47:54.562327   20   
4        5  Toronto  91.719953    True 2023-04-06 21:47:54.562327   67   

   sessions     revenue  
0       323  179.164861  
1       352  614.893856  
2       279  217.069799  
3       129  195.459273  
4       199  617.349590  
500000


In [2]:
import pyarrow as pa
import pyarrow.parquet as pq

table = pa.Table.from_pandas(df)
pq.write_table(table, "users.parquet")

pf = pq.ParquetFile("users.parquet")

print(pf.num_row_groups)
print(pf.metadata.num_rows)
print(pf.metadata.num_columns)
print(pf.schema)

rg = pf.metadata.row_group(0)

for i in range(rg.num_columns):
    col = rg.column(i)
    stats = col.statistics
    print(col.path_in_schema)
    print(col.physical_type)
    print(col.total_compressed_size)
    if stats:
        print(stats.min, stats.max, stats.null_count)

row_group = pf.metadata.row_group(0)

for i in range(row_group.num_columns):
    col = row_group.column(i)
    print("Column:", col.path_in_schema)
    print("Physical type:", col.physical_type)
    print("Compressed size:", col.total_compressed_size)
    print("Stats:", col.statistics)

1
500000
8
required group field_id=-1 schema {
  optional int64 field_id=-1 user_id;
  optional binary field_id=-1 city (String);
  optional double field_id=-1 score;
  optional boolean field_id=-1 active;
  optional int64 field_id=-1 signup_date (Timestamp(isAdjustedToUTC=false, timeUnit=nanoseconds, is_from_converted_type=false, force_set_converted_type=false));
  optional int32 field_id=-1 age;
  optional int32 field_id=-1 sessions;
  optional double field_id=-1 revenue;
}

user_id
INT64
2280081
1 500000 0
city
BYTE_ARRAY
251173
Berlin Toronto 0
score
DOUBLE
4279334
5.188445665327279e-05 99.99983148609545 0
active
BOOLEAN
62553
False True 0
signup_date
INT64
697197
2023-03-11 21:47:54.562327 2026-03-09 21:47:54.562327 0
age
INT32
376346
18 80 0
sessions
INT32
565611
0 500 0
revenue
DOUBLE
4279334
0.0009958058022618843 999.9978869129644 0
Column: user_id
Physical type: INT64
Compressed size: 2280081
Stats: <pyarrow._parquet.Statistics object at 0x0000029D6BA49FD0>
  has_min_max: True

In [3]:
import os

df.to_csv("users.csv", index=False)

parquet_size = os.path.getsize("users.parquet")
csv_size = os.path.getsize("users.csv")

parquet_kb = parquet_size / 1024
csv_kb = csv_size / 1024

ratio = csv_size / parquet_size

print(parquet_kb)
print(csv_kb)
print(ratio)

12496.4140625
44044.9091796875
3.5246038551059335


Parquet metadata contains useful information about the data structure that CSV does not store. For example, it keeps the schema (column names and data types), number of rows, row groups, and statistics like min, max, and null count for each column.

CSV files only store raw text data without this metadata. Because of this, systems must read the whole file to understand the data.

Parquet metadata helps query engines skip unnecessary data and read only the needed parts. This makes queries faster and more efficient, especially for large datasets.

<h1>TASK-2</h1>

In [4]:
import time
import pyarrow.parquet as pq

start = time.time()

table = pq.read_table("users.parquet")
df_full = table.to_pandas()

end = time.time()

print(end - start)

0.050679922103881836


In [5]:

start = time.time()

table = pq.read_table("users.parquet", columns=["user_id", "revenue"])
df_two = table.to_pandas()

end = time.time()

print(end - start)

0.01264190673828125


In [6]:

start = time.time()

df_csv = pd.read_csv("users.csv")
df_two_csv = df_csv[["user_id", "revenue"]]

end = time.time()

print(end - start)

0.2821357250213623


Parquet reads are faster when selecting only a few columns because of its column-chunk layout. In Parquet, data for each column is stored separately inside row groups. This means the system can read only the needed columns without touching the others.

In Task 1 we saw that each column has its own metadata and storage inside the row group. Because of this structure, when we request only two columns, Parquet loads only those column chunks.

CSV files are stored row by row as plain text. When pandas reads a CSV file, it must scan the entire file first and then select the needed columns. This extra work makes CSV slower for selective reads, especially with large datasets.

<h1>TASK-3</h1>

In [7]:

start = time.time()

table = pq.read_table("users.parquet", filters=[("age", ">", 50)])
df_filtered = table.to_pandas()

end = time.time()

print(end - start)
print(len(df_filtered))

0.033330440521240234
237812


In [8]:

start = time.time()

table = pq.read_table("users.parquet")
df_all = table.to_pandas()
df_filtered = df_all[df_all["age"] > 50]

end = time.time()

print(end - start)
print(len(df_filtered))

0.042708635330200195
237812


Predicate pushdown is a feature in Parquet that applies filters while reading the file, instead of after loading all the data. For example, when we used age > 50 in PyArrow, only the relevant row groups and column chunks were read from disk.

When we load the full Parquet file first and filter in pandas, the system reads all 500,000 rows before applying the filter. This takes more time and uses more memory.

Predicate pushdown is faster because it avoids reading unnecessary rows and reduces disk I/O, making queries on large datasets much more efficient.

In [9]:
import duckdb

start = time.time()

result = duckdb.sql("SELECT * FROM 'users.parquet' WHERE age > 50").df()

end = time.time()

print(end - start)
print(len(result))

0.04264116287231445
237812


<h1>TASK-4</h1>

In [10]:
start = time.time()
city_counts_pd = df.groupby("city").size()
end = time.time()
print("Pandas:", end - start)
print(city_counts_pd)

start = time.time()
city_counts_duck = duckdb.sql("SELECT city, COUNT(*) AS count FROM 'users.parquet' GROUP BY city").df()
end = time.time()
print("DuckDB:", end - start)
print(city_counts_duck)

Pandas: 0.02102375030517578
city
Berlin      50059
Dubai       50059
London      50216
Madrid      49724
New York    50113
Paris       49931
Rome        50061
Sydney      49885
Tokyo       49970
Toronto     49982
dtype: int64
DuckDB: 0.004453897476196289
       city  count
0    Sydney  49885
1      Rome  50061
2    Berlin  50059
3     Dubai  50059
4    Madrid  49724
5     Tokyo  49970
6     Paris  49931
7  New York  50113
8   Toronto  49982
9    London  50216


In [11]:
start = time.time()
avg_score_pd = df.groupby("city")["score"].mean().sort_values(ascending=False)
end = time.time()
print("Pandas:", end - start)
print(avg_score_pd)

start = time.time()
avg_score_duck = duckdb.sql("""
    SELECT city, AVG(score) AS avg_score
    FROM 'users.parquet'
    GROUP BY city
    ORDER BY avg_score DESC
""").df()
end = time.time()
print("DuckDB:", end - start)
print(avg_score_duck)

Pandas: 0.019583463668823242
city
Dubai       50.311448
London      50.151899
Tokyo       50.135854
Toronto     50.075511
Sydney      50.063780
Berlin      50.013372
Paris       50.009568
Madrid      49.996347
Rome        49.994487
New York    49.879502
Name: score, dtype: float64
DuckDB: 0.0071430206298828125
       city  avg_score
0     Dubai  50.311448
1    London  50.151899
2     Tokyo  50.135854
3   Toronto  50.075511
4    Sydney  50.063780
5    Berlin  50.013372
6     Paris  50.009568
7    Madrid  49.996347
8      Rome  49.994487
9  New York  49.879502


In [12]:
start = time.time()
active_high_pd = df[df["score"] > 75].groupby("city")["active"].mean() * 100
end = time.time()
print("Pandas:", end - start)
print(active_high_pd)

start = time.time()
active_high_duck = duckdb.sql("""
    SELECT city,
           100.0 * SUM(CASE WHEN active THEN 1 ELSE 0 END) / COUNT(*) AS pct_active_high
    FROM 'users.parquet'
    WHERE score > 75
    GROUP BY city
""").df()
end = time.time()
print("DuckDB:", end - start)
print(active_high_duck)

Pandas: 0.014236927032470703
city
Berlin      50.056126
Dubai       50.051193
London      49.717739
Madrid      49.695415
New York    49.444044
Paris       50.466396
Rome        50.152586
Sydney      50.084303
Tokyo       50.479004
Toronto     50.067691
Name: active, dtype: float64
DuckDB: 0.01171875
       city  pct_active_high
0    Sydney        50.084303
1      Rome        50.152586
2    Berlin        50.056126
3     Tokyo        50.479004
4    Madrid        49.695415
5   Toronto        50.067691
6    London        49.717739
7     Dubai        50.051193
8     Paris        50.466396
9  New York        49.444044


In [13]:

start = time.time()
df["rank"] = df.groupby("city")["score"].rank(method="first", ascending=False)
top10_pd = df[df["rank"] <= 10].sort_values(["city", "rank"])
end = time.time()
print("Pandas:", end - start)
print(top10_pd[["city", "user_id", "score", "rank"]])

start = time.time()
top10_duck = duckdb.sql("""
    SELECT *
    FROM (
        SELECT *,
               RANK() OVER (PARTITION BY city ORDER BY score DESC) AS rank
        FROM 'users.parquet'
    ) sub
    WHERE rank <= 10
    ORDER BY city, rank
""").df()
end = time.time()
print("DuckDB:", end - start)
print(top10_duck)

Pandas: 0.15570998191833496
           city  user_id      score  rank
222164   Berlin   222165  99.994823   1.0
474291   Berlin   474292  99.992728   2.0
96404    Berlin    96405  99.992615   3.0
348899   Berlin   348900  99.992278   4.0
198655   Berlin   198656  99.988107   5.0
...         ...      ...        ...   ...
207207  Toronto   207208  99.994039   6.0
139064  Toronto   139065  99.993684   7.0
245209  Toronto   245210  99.992284   8.0
344537  Toronto   344538  99.990280   9.0
479832  Toronto   479833  99.987232  10.0

[100 rows x 4 columns]
DuckDB: 0.08338809013366699
    user_id     city      score  active                signup_date  age  \
0    222165   Berlin  99.994823    True 2025-05-04 21:47:54.562327   75   
1    474292   Berlin  99.992728    True 2025-01-06 21:47:54.562327   48   
2     96405   Berlin  99.992615   False 2024-07-17 21:47:54.562327   57   
3    348900   Berlin  99.992278   False 2024-12-21 21:47:54.562327   67   
4    198656   Berlin  99.988107   False 2

In [14]:
start = time.time()
df["running_total"] = df.groupby("city")["score"].cumsum()
end = time.time()
print("Pandas:", end - start)
print(df[["city", "user_id", "score", "running_total"]].head(20))

start = time.time()
running_total_duck = duckdb.sql("""
    SELECT *,
           SUM(score) OVER (PARTITION BY city ORDER BY user_id) AS running_total
    FROM 'users.parquet'
""").df()
end = time.time()
print("DuckDB:", end - start)
print(running_total_duck.head(20))

Pandas: 0.02160930633544922
        city  user_id      score  running_total
0    Toronto        1  43.689490      43.689490
1      Paris        2  97.403462      97.403462
2     Sydney        3  66.148238      66.148238
3     London        4  21.993544      21.993544
4    Toronto        5  91.719953     135.409443
5       Rome        6  60.497154      60.497154
6   New York        7  41.789961      41.789961
7    Toronto        8  27.220312     162.629755
8     Sydney        9   9.353333      75.501571
9     London       10  39.357197      61.350742
10     Paris       11  55.135777     152.539239
11    Sydney       12  77.267889     152.769459
12    Sydney       13  61.616970     214.386429
13  New York       14  73.611226     115.401187
14     Dubai       15  22.197637      22.197637
15    London       16  67.966054     129.316795
16     Tokyo       17  50.925029      50.925029
17    Sydney       18   8.859391     223.245820
18     Dubai       19  19.583048      41.780685
19     Tokyo

Comparison of pandas vs DuckDB

Ease of use:
Pandas is straightforward for simple aggregations like counting or averaging, but queries involving window functions (rank, running totals) are more complex to write. DuckDB uses SQL, so even advanced queries are concise and readable.

Performance:
DuckDB was generally faster, especially for queries with filtering, grouping, and window functions. Pandas had to load the full dataset into memory, which slowed down large operations.

Queries where difference mattered most:
Queries 4 (top 10 users per city) and 5 (running total per city) showed the biggest speed difference. DuckDB handled these efficiently on disk, while pandas required heavy in-memory computation, making it noticeably slower.

<h1>TASK-5</h1>

In [15]:
df_sample = pd.DataFrame({
    "user_id": np.arange(1, 11),
    "city": ["Berlin","Tokyo","New York","Paris","London","Dubai","Toronto","Sydney","Madrid","Rome"],
    "score": np.random.uniform(0, 100, 10),
    "active": np.random.choice([True, False], 10)
})

arrow_table = pa.Table.from_pandas(df_sample)
print(arrow_table)

pyarrow.Table
user_id: int64
city: string
score: double
active: bool
----
user_id: [[1,2,3,4,5,6,7,8,9,10]]
city: [["Berlin","Tokyo","New York","Paris","London","Dubai","Toronto","Sydney","Madrid","Rome"]]
score: [[87.73718439362575,42.1762118569052,68.26312800534747,79.40535796134546,43.543520613349585,68.02575630953619,87.5200719391783,22.428605702668737,1.0174318238478808,87.37078001689363]]
active: [[false,true,true,true,false,false,true,true,true,true]]


In [16]:
print(df_sample.dtypes)

print(arrow_table.schema)

user_id      int64
city        object
score      float64
active        bool
dtype: object
user_id: int64
city: string
score: double
active: bool
-- schema metadata --
pandas: '{"index_columns": [{"kind": "range", "name": null, "start": 0, "' + 697


In [17]:
import pyarrow.parquet as pq

pq.write_table(arrow_table, "sample_arrow.parquet")

arrow_table_read = pq.read_table("sample_arrow.parquet")
print(arrow_table_read)

pyarrow.Table
user_id: int64
city: string
score: double
active: bool
----
user_id: [[1,2,3,4,5,6,7,8,9,10]]
city: [["Berlin","Tokyo","New York","Paris","London","Dubai","Toronto","Sydney","Madrid","Rome"]]
score: [[87.73718439362575,42.1762118569052,68.26312800534747,79.40535796134546,43.543520613349585,68.02575630953619,87.5200719391783,22.428605702668737,1.0174318238478808,87.37078001689363]]
active: [[false,true,true,true,false,false,true,true,true,true]]


In [18]:
df_from_arrow = arrow_table_read.to_pandas()

print(df_from_arrow)
print(df_from_arrow.equals(df_sample))

   user_id      city      score  active
0        1    Berlin  87.737184   False
1        2     Tokyo  42.176212    True
2        3  New York  68.263128    True
3        4     Paris  79.405358    True
4        5    London  43.543521   False
5        6     Dubai  68.025756   False
6        7   Toronto  87.520072    True
7        8    Sydney  22.428606    True
8        9    Madrid   1.017432    True
9       10      Rome  87.370780    True
True


In [19]:
df_arrow_backed = pd.read_parquet("sample_arrow.parquet", dtype_backend="pyarrow")


print(df_arrow_backed.dtypes)

df_traditional = pd.read_parquet("sample_arrow.parquet")
print(df_traditional.dtypes)

user_id     int64[pyarrow]
city       string[pyarrow]
score      double[pyarrow]
active       bool[pyarrow]
dtype: object
user_id      int64
city        object
score      float64
active        bool
dtype: object


The Role of Arrow in Modern Analytics

Arrow acts as a fast, in-memory bridge between different tools in the analytics stack. Parquet stores data efficiently on disk in a columnar format. When data is read into pandas, Arrow allows it to be represented with zero-copy memory, preserving the columnar structure. DuckDB can also consume Arrow Tables directly, enabling SQL queries without full data conversion.

This means Arrow connects storage (Parquet), analysis (pandas), and SQL engines (DuckDB) efficiently, reducing memory usage, speeding up data transfers, and making complex workflows seamless.